In [1]:
# Generate the stats of the keywords evaluated
import os
import json
from dotenv import load_dotenv
from sqlalchemy import create_engine, select, update
from sqlalchemy.orm import sessionmaker
from models import DS_Generated_Review_Final, DS_Commit


In [2]:
load_dotenv()

db_host = os.getenv('db_host')
db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

In [3]:
DATABASE_URL = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)

In [4]:
with open('security_keywords.txt', 'r') as f:
    security_keywords = [line.strip() for line in f.readlines() if line.strip()]

print(security_keywords)

['TOCTOU', 'breach', 'code injection', 'cross site', 'ddos', 'dead lock', 'dead-lock', 'deadlock', 'deadlocks', 'denial of service', 'exploit', 'forged', 'gain access', 'infinite recursion', 'infinite-recursion', 'leak', 'malicious', 'malicious code', 'overflow', 'overrun', 'race', 'races', 'racy', 'redos', 'sql injection', 'stack overflow', 'unauthenticated', 'underflow', 'vulnerability', 'vulnerable', 'xsrf', 'xss', 'xxe', 'zip slip', 'zipslip']


In [5]:
usable_commits = []
with Session() as session:
    # Query the database for usable commits
    qry = select(DS_Generated_Review_Final.sha).where(DS_Generated_Review_Final.usable == True)
    usable_commits = session.execute(qry).scalars().all()

print(f"Total usable commits: {len(usable_commits)}")


Total usable commits: 17188


In [6]:
with Session() as session:
    # Query the database for commit messages of usable commits
    qry = select(DS_Commit).where(DS_Commit.sha.in_(usable_commits))
    commits = session.execute(qry).scalars().all()
    commit_messages = {commit.sha: {'msg':commit.message, 'keywords': commit.keywords} for commit in commits}

In [7]:
# Count the occurrences of each keyword in the commit messages
keyword_stats = {keyword: 0 for keyword in security_keywords}
keyword_stats_keywords = {keyword: 0 for keyword in security_keywords}
for sha, data in commit_messages.items():
    message = data['msg'].lower()
    message_keywords = data['keywords'].lower()
    for keyword in security_keywords:
        msg_bool = False
        kw_bool = False
        if keyword.lower() in message:
            keyword_stats[keyword] += 1
            msg_bool = True
        
        if keyword.lower() in message_keywords:
            keyword_stats_keywords[keyword] += 1
            kw_bool = True
        
        # if msg_bool != kw_bool:
        #     print(f"Discrepancy for commit {sha} on keyword '{keyword}': in message={msg_bool}, in keywords={kw_bool}")
        #     print(f"  Message: {message}")
        #     print(f"  Keywords: {message_keywords}")

In [8]:
#Check for discrepancies
for keyword in security_keywords:
    count_msg = keyword_stats[keyword]
    count_kw = keyword_stats_keywords[keyword]
    if count_msg != count_kw:
        print(f"Discrepancy for keyword '{keyword}': in messages={count_msg}, in keywords={count_kw}")

Discrepancy for keyword 'dead lock': in messages=108, in keywords=106
Discrepancy for keyword 'dead-lock': in messages=27, in keywords=26
Discrepancy for keyword 'deadlock': in messages=2695, in keywords=2693
Discrepancy for keyword 'exploit': in messages=232, in keywords=222
Discrepancy for keyword 'infinite recursion': in messages=301, in keywords=300
Discrepancy for keyword 'leak': in messages=4929, in keywords=4895
Discrepancy for keyword 'malicious': in messages=191, in keywords=189
Discrepancy for keyword 'overflow': in messages=2331, in keywords=2291
Discrepancy for keyword 'race': in messages=5492, in keywords=5398
Discrepancy for keyword 'races': in messages=198, in keywords=164
Discrepancy for keyword 'racy': in messages=71, in keywords=65
Discrepancy for keyword 'redos': in messages=6, in keywords=5
Discrepancy for keyword 'sql injection': in messages=62, in keywords=61
Discrepancy for keyword 'stack overflow': in messages=507, in keywords=506


In [9]:
# Order the dicts from higher to lower on the values
keyword_stats_ordered = dict(sorted(keyword_stats.items(), key=lambda kv: kv[1], reverse=True))
keyword_stats_keywords_ordered = dict(sorted(keyword_stats_keywords.items(), key=lambda kv: kv[1], reverse=True))

# Print top 20 for quick inspection
print("Top keywords in commit messages:")
for k, v in list(keyword_stats_ordered.items())[:20]:
    print(f"{k}: {v}")

print("\nTop keywords in keywords field:")
for k, v in list(keyword_stats_keywords_ordered.items())[:20]:
    print(f"{k}: {v}")

Top keywords in commit messages:
race: 5492
leak: 4929
deadlock: 2695
overflow: 2331
vulnerability: 545
stack overflow: 507
deadlocks: 372
infinite recursion: 301
xss: 268
exploit: 232
races: 198
malicious: 191
vulnerable: 124
dead lock: 108
xxe: 92
underflow: 74
racy: 71
sql injection: 62
unauthenticated: 53
zip slip: 48

Top keywords in keywords field:
race: 5398
leak: 4895
deadlock: 2693
overflow: 2291
vulnerability: 545
stack overflow: 506
deadlocks: 372
infinite recursion: 300
xss: 268
exploit: 222
malicious: 189
races: 164
vulnerable: 124
dead lock: 106
xxe: 92
underflow: 74
racy: 65
sql injection: 61
unauthenticated: 53
zip slip: 48


In [10]:
# print("Keyword stats in commit messages:")
# for keyword, count in keyword_stats.items():
#     print(f"{keyword}: {count}")

In [11]:
print("Keyword stats in keywords field:")
for keyword, count in keyword_stats_keywords_ordered.items():
    print(f"{keyword}: {count}")

Keyword stats in keywords field:
race: 5398
leak: 4895
deadlock: 2693
overflow: 2291
vulnerability: 545
stack overflow: 506
deadlocks: 372
infinite recursion: 300
xss: 268
exploit: 222
malicious: 189
races: 164
vulnerable: 124
dead lock: 106
xxe: 92
underflow: 74
racy: 65
sql injection: 61
unauthenticated: 53
zip slip: 48
malicious code: 31
dead-lock: 26
zipslip: 26
cross site: 17
overrun: 17
TOCTOU: 13
denial of service: 12
xsrf: 11
code injection: 8
ddos: 6
redos: 5
breach: 2
gain access: 2
infinite-recursion: 2
forged: 1


In [12]:
# top 4 keywords
top_keywords = list(keyword_stats_keywords_ordered.keys())[:4]
print(f"Top 4 keywords: {top_keywords}")

Top 4 keywords: ['race', 'leak', 'deadlock', 'overflow']


In [15]:
shas_to_upsample = []
shas_to_downsample = []
shas_to_keep = []
for sha, data in commit_messages.items():
    message_keywords = data['keywords']
    message_keywords = message_keywords.split(',')
    # If all keywords in the message are in the top keywords, downsample
    if all(kw.strip() in top_keywords for kw in message_keywords):
        shas_to_downsample.append(sha)
    elif any(kw.strip() in top_keywords for kw in message_keywords):
        shas_to_keep.append(sha)
    else:
        shas_to_upsample.append(sha)

In [16]:
print(f"Commits to downsample (contain top keywords): {len(shas_to_downsample)}")
print(f"Commits to upsample (do not contain top keywords): {len(shas_to_upsample)}")
print(f"Commits to keep (contain some but not all top keywords): {len(shas_to_keep)}")

Commits to downsample (contain top keywords): 13961
Commits to upsample (do not contain top keywords): 2439
Commits to keep (contain some but not all top keywords): 788


In [18]:
shas_to_keep

['...']

In [19]:
shas_to_downsample

['...',
 ...]

In [20]:
info = {
    'shas_to_downsample': shas_to_downsample,
    'shas_to_upsample': shas_to_upsample,
    'shas_to_keep': shas_to_keep
}

with open('keyword_sampling_info.json', 'w') as f:
    json.dump(info, f)